# **Child Well-being - Data Partitioner**<br/>
**University: University of Milano-Bicocca**<br/>
**Master's Degree: Data Science (A.Y. 2025/2026)**<br/>
**Course: Data Science Lab**<br/>

In [1]:
import sys
sys.path.insert(0, "..")

import polars as pl

from scripts.transformer import discretize, cascade_aggregate, cascade_level2

In [2]:
INDICATORS: dict[str, dict] = {
    # ===== A — Outcomes =====
    # A1 — Material conditions
    "A1_2": {
        "description": "Severe housing deprivation (0-17 years)",
        "type": "indicator",
        "dimension": "A",
        "subdimension": "A1_material_conditions",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "0-17",
        "source": "Eurostat / EU-SILC",
    },
    # A2 — Physical health conditions
    "A2_1": {
        "description": "Infant mortality (under 1 year)",
        "type": "indicator",
        "dimension": "A",
        "subdimension": "A2_health",
        "measure_type": "rate",
        "direction": "negative",
        "age_group": "<1",
        "source": "OECD Health Statistics",
    },
    # A3 — Cognitive & educational outcome
    "A3_3": {
        "description": "Top performers PISA (level 5-6 in at least one subject)",
        "type": "indicator",
        "dimension": "A",
        "subdimension": "A3_education",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    "A3_4": {
        "description": "Students who expect to complete tertiary education",
        "type": "indicator",
        "dimension": "A",
        "subdimension": "A3_education",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    "A3_5": {
        "description": "NEET (15-29 years)",
        "type": "indicator",
        "dimension": "A",
        "subdimension": "A3_education",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "15-29",
        "source": "OECD / Eurostat",
    },
    # A4 — Social & emotional outcome
    "A4_6": {
        "description": "High life satisfaction (score 9-10)",
        "type": "indicator",
        "dimension": "A",
        "subdimension": "A4_subjective_wellbeing",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },

    # ===== B — Relationships and Social Context =====
    # B1 — Home & Family Life
    "B1_1": {
        "description": "Relative child poverty (50% median threshold)",
        "type": "indicator",
        "dimension": "B",
        "subdimension": "B1_family_income",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "0-17",
        "source": "OECD / EU-SILC",
    },
    "B1_5": {
        "description": "Parents encourage self-confidence (strongly agree)",
        "type": "indicator",
        "dimension": "B",
        "subdimension": "B1_family_income",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    # B2 — Life at school & early education and care
    "B2_4": {
        "description": "Bullying at school (at least once a month)",
        "type": "indicator",
        "dimension": "B",
        "subdimension": "B2_school_early_childhood",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "15",
        "source": "PISA",
    },
    "B2_5": {
        "description": "Sense of belonging to school (agree/strongly agree)",
        "type": "indicator",
        "dimension": "B",
        "subdimension": "B2_school_early_childhood",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    # B3 — Social Life and life in the community
    "B3_5": {
        "description": "Criminality/violence in the area of residence",
        "type": "indicator",
        "dimension": "B",
        "subdimension": "B3_safety",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "0-17",
        "source": "EU-SILC",
    },
    # B4 — Online life
    "B4_3": {
        "description": "Internet as a major source of information (strongly agree)",
        "type": "indicator",
        "dimension": "B",
        "subdimension": "B4_digital",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },

    # ===== C — Policies and Resources =====
    "C1_2": {
        "description": "Reduction in child poverty from taxes/transfers (pp)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "percentage_points",
        "direction": "positive",
        "age_group": "0-17",
        "source": "OECD",
    },
    "C1_3": {
        "description": "Minimum guaranteed income for couples with 2 children (% median)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Tax-Benefit Model",
    },
    "C1_4": {
        "description": "Maternity leave with pay (total weeks)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "weeks",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Family Database",
    },
    "C1_5": {
        "description": "Paternity leave with pay (total weeks)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "weeks",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Family Database",
    },
    # C2 — House & Community policies
    "C2_1": {
        "description": "Public expenditure on housing and community services per person",
        "type": "public_expenditure",
        "dimension": "C",
        "subdimension": "C2_housing_culture",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
    "C2_2": {
        "description": "Public expenditure on recreation, culture and religion per person",
        "type": "public_expenditure",
        "dimension": "C",
        "subdimension": "C2_housing_culture",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
    "C2_3": {
        "description": "Public expenditure on social protection and housing per person",
        "type": "public_expenditure",
        "dimension": "C",
        "subdimension": "C2_housing_culture",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
    # C3 — Health policies
    "C3_1": {
        "description": "Public expenditure on public health per person",
        "type": "public_expenditure",
        "dimension": "C",
        "subdimension": "C3_public_health",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Health Statistics",
    },
    "C3_2": {
        "description": "Vaccination DPT 3 doses (% under 1 year)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C3_public_health",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "<1",
        "source": "OECD Health Statistics",
    },
    "C3_3": {
        "description": "Vaccination measles at least 1 dose (% under 1 year)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C3_public_health",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "<1",
        "source": "OECD Health Statistics",
    },
    # C4 — Education & early childhood policies
    "C4_2": {
        "description": "Net cost of childcare per parent (low earning couple, 2 children)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "n/a",
        "source": "OECD Tax-Benefit Model",
    },
    "C4_4": {
        "description": "Public expenditure on primary and secondary education per student FTE",
        "type": "public_expenditure",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "primary-secondary",
        "source": "OECD Education at a Glance",
    },
    "C4_5": {
        "description": "Public expenditure on ancillary education services per student FTE",
        "type": "public_expenditure",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "primary-secondary",
        "source": "OECD Education at a Glance",
    },
    "C4_6": {
        "description": "Student-teacher ratio secondary (FTE)",
        "type": "indicator",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "ratio",
        "direction": "negative",
        "age_group": "secondary",
        "source": "OECD Education at a Glance",
    },
    # C5 — Environmental policies
    "C5_1": {
        "description": "Public expenditure on environment per person",
        "type": "public_expenditure",
        "dimension": "C",
        "subdimension": "C5_environment",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
}

ALL_INDICATORS = list(INDICATORS.keys())

NEGATIVE_INDICATORS = [k for k, v in INDICATORS.items() if v["direction"] == "negative"]
POSITIVE_INDICATORS = [k for k, v in INDICATORS.items() if v["direction"] == "positive"]

DIMENSIONS = sorted(set(v["dimension"] for v in INDICATORS.values()))
SUBDIMENSIONS = sorted(set(v["subdimension"] for v in INDICATORS.values()))

INDICATORS_BY_DIMENSION = {
    dim: [k for k, v in INDICATORS.items() if v["dimension"] == dim]
    for dim in DIMENSIONS
}

INDICATORS_BY_SUBDIMENSION = {
    sub: [k for k, v in INDICATORS.items() if v["subdimension"] == sub]
    for sub in SUBDIMENSIONS
}

print(f"Total indicators: {len(ALL_INDICATORS)}")
print(f"  - positive direction: {len(POSITIVE_INDICATORS)}")
print(f"  - negative direction: {len(NEGATIVE_INDICATORS)}")
print(f"Dimensions: {DIMENSIONS}")
print(f"Sub-dimensions: {SUBDIMENSIONS}")
print()

for dim, cols in INDICATORS_BY_DIMENSION.items():
    print(f"[{dim}] ({len(cols)} indicators): {cols}")

print("\nIndicators to invert:")
for ind in NEGATIVE_INDICATORS:
    print(f"  {ind}: {INDICATORS[ind]['description']}")

print("\nIndicators to keep:")
for ind in POSITIVE_INDICATORS:
    print(f"  {ind}: {INDICATORS[ind]['description']}")

Total indicators: 27
  - positive direction: 19
  - negative direction: 8
Dimensions: ['A', 'B', 'C']
Sub-dimensions: ['A1_material_conditions', 'A2_health', 'A3_education', 'A4_subjective_wellbeing', 'B1_family_income', 'B2_school_early_childhood', 'B3_safety', 'B4_digital', 'C1_family_spending', 'C2_housing_culture', 'C3_public_health', 'C4_education_childcare', 'C5_environment']

[A] (6 indicators): ['A1_2', 'A2_1', 'A3_3', 'A3_4', 'A3_5', 'A4_6']
[B] (6 indicators): ['B1_1', 'B1_5', 'B2_4', 'B2_5', 'B3_5', 'B4_3']
[C] (15 indicators): ['C1_2', 'C1_3', 'C1_4', 'C1_5', 'C2_1', 'C2_2', 'C2_3', 'C3_1', 'C3_2', 'C3_3', 'C4_2', 'C4_4', 'C4_5', 'C4_6', 'C5_1']

Indicators to invert:
  A1_2: Severe housing deprivation (0-17 years)
  A2_1: Infant mortality (under 1 year)
  A3_5: NEET (15-29 years)
  B1_1: Relative child poverty (50% median threshold)
  B2_4: Bullying at school (at least once a month)
  B3_5: Criminality/violence in the area of residence
  C4_2: Net cost of childcare per par

In [3]:
df = pl.read_parquet('../data/030_child_well_being.parquet')

---

### 1. Divide Data into Dataframes by year and indicator_type (indicator/public_expenditure)

In [5]:
public_expenditure_columns = [
    "C2_1", "C2_2", "C2_3",
    "C3_1", 
    "C4_4", "C4_5", 
    "C5_1"
]
base_columns = [x for x in df.columns if x not in public_expenditure_columns]
public_expenditure_columns.extend(['TIME_PERIOD', 'REF_AREA'])

In [6]:
df_ind_2015 = df.filter(
    pl.col('TIME_PERIOD') == 2015
)[base_columns]
df_ind_2018 = df.filter(
    pl.col('TIME_PERIOD') == 2018
)[base_columns]

df_pe_2015 = df.filter(
    pl.col('TIME_PERIOD') == 2015
)[public_expenditure_columns]
df_pe_2018 = df.filter(
    pl.col('TIME_PERIOD') == 2018
)[public_expenditure_columns]

In [7]:
df_for_poset = {
    "indicators_2015": df_ind_2015,
    "indicators_2018": df_ind_2018,
    "public_expenditure_2015": df_pe_2015,
    "public_expenditure_2018": df_pe_2018,
}

### 2. Create 4 POSets based on the 11 OECD Dimensions

In [8]:
results = {}

for name, df in df_for_poset.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}\n")
    
    results[name] = cascade_aggregate(
        df=df,
        indicators=INDICATORS,
        col1="REF_AREA",
        col2="TIME_PERIOD",
        mrp_mode="approximate", # use approximate MRP for faster computation (comparing to exact MRP which is more accurate but much slower)
        seed=42,
        verbose=True,
    )


  indicators_2015

Available indicators: 20 / 27
  ['A1_2', 'A2_1', 'A3_3', 'A3_4', 'A3_5', 'A4_6', 'B1_1', 'B1_5', 'B2_4', 'B2_5', 'B3_5', 'B4_3', 'C1_2', 'C1_3', 'C1_4', 'C1_5', 'C3_2', 'C3_3', 'C4_2', 'C4_6']
{('indicator', 'A1_material_conditions'): ['A1_2'], ('indicator', 'A2_health'): ['A2_1'], ('indicator', 'A3_education'): ['A3_3', 'A3_4', 'A3_5'], ('indicator', 'A4_subjective_wellbeing'): ['A4_6'], ('indicator', 'B1_family_income'): ['B1_1', 'B1_5'], ('indicator', 'B2_school_early_childhood'): ['B2_4', 'B2_5'], ('indicator', 'B3_safety'): ['B3_5'], ('indicator', 'B4_digital'): ['B4_3'], ('indicator', 'C1_family_spending'): ['C1_2', 'C1_3', 'C1_4', 'C1_5'], ('indicator', 'C3_public_health'): ['C3_2', 'C3_3'], ('indicator', 'C4_education_childcare'): ['C4_2', 'C4_6']}
Units: 16
Available indicators: 20
Groups: 11
  [indicator] A1_material_conditions: ['A1_2'] → pass-through
  [indicator] A2_health: ['A2_1'] → pass-through
  [indicator] A3_education: ['A3_3', 'A3_4', 'A3_5'] → s

In [9]:
def apply_discretization(df: pl.DataFrame, columns: list[str], n_levels: int = 4, normalized_suffix: str = "_norm") -> pl.DataFrame:
    df =  discretize(
        df,
        columns=columns,
        n_levels=n_levels,
        normalized_suffix=normalized_suffix,
    )

    df = df.drop([col for col in df.columns if col.endswith("_norm") or col in columns]).rename(
        {col: col.replace("_ord", "") for col in df.columns if col.endswith("_ord")}
    )
    return df

In [13]:
n_levels = 4
df_ind_2015 = apply_discretization(results['indicators_2015']['indicator'], columns=SUBDIMENSIONS, n_levels=n_levels, normalized_suffix="_norm")
df_ind_2018 = apply_discretization(results['indicators_2018']['indicator'], columns=SUBDIMENSIONS, n_levels=n_levels, normalized_suffix="_norm")
df_pe_2015 = apply_discretization(results['public_expenditure_2015']['public_expenditure'], columns=SUBDIMENSIONS, n_levels=n_levels, normalized_suffix="_norm")
df_pe_2018 = apply_discretization(results['public_expenditure_2018']['public_expenditure'], columns=SUBDIMENSIONS, n_levels=n_levels, normalized_suffix="_norm")

In [14]:
df_ind_2015.write_parquet(f'../data/040_indicators_dim_discrete_{n_levels}_level_2015.parquet')
df_ind_2018.write_parquet(f'../data/040_indicators_dim_discrete_{n_levels}_level_2018.parquet')
df_pe_2015.write_parquet(f'../data/040_public_expenditure_dim_discrete_{n_levels}_level_2015.parquet')
df_pe_2018.write_parquet(f'../data/040_public_expenditure_dim_discrete_{n_levels}_level_2018.parquet')

The output of the first aggregation does not yield valuable results: in fact, the POSets based on the 11 dimensions of the indicators do not represent any dominance between countries for both of the years

---

### 3. Grouping 11 Dimensions in 4 Macro(-Logical)-Dimensions

We keep 4 levels for each dimension in order to avoid excessive data compression and to obtain more stable results.

In [15]:
n_levels = 4
df_ind_2015 = pl.read_parquet(f'../data/040_indicators_dim_discrete_{n_levels}_level_2015.parquet')
df_ind_2018 = pl.read_parquet(f'../data/040_indicators_dim_discrete_{n_levels}_level_2018.parquet')
df_exp_2015 = pl.read_parquet(f'../data/040_public_expenditure_dim_discrete_{n_levels}_level_2015.parquet')
df_exp_2018 = pl.read_parquet(f'../data/040_public_expenditure_dim_discrete_{n_levels}_level_2018.parquet')

In [16]:
LEVEL2_GROUPS = {
    "material_protection": ["A1_material_conditions", "B1_family_income", "C1_family_spending"],
    "health_prevention": ["A2_health", "C3_public_health"],
    "education_skills": ["A3_education", "C4_education_childcare"],
    "social_wellbeing": ["A4_subjective_wellbeing", "B2_school_early_childhood", "B3_safety", "B4_digital"],
}

level2 = {}
for year, df in {"2015": df_ind_2015, "2018": df_ind_2018}.items():
    print(f"\n{'='*60}")
    print(f"  LEVEL 2 — INDICATORS {year}")
    print(f"{'='*60}\n")
    
    level2[year] = cascade_level2(
        df=df,
        groups=LEVEL2_GROUPS,
        col1="REF_AREA",
        col2="TIME_PERIOD",
        n_levels=3,
        mrp_mode="exact", # now we prefer using exact MRP for level 2 as we have much fewer columns (4 sub-dimensions aggregated into
        # 4 level 2 groups) so it should be computationally feasible and more accurate than approximate MRP
        seed=42,
        verbose=True,
    )

df_macro_2015 = level2["2015"]["aggregated"]
df_macro_2018 = level2["2018"]["aggregated"]

print("\n2015:")
print(df_macro_2015)
print("\n2018:")
print(df_macro_2018)


  LEVEL 2 — INDICATORS 2015

Level 2 cascade: 16 units
Macro-dimensions: 4
  material_protection: ['A1_material_conditions', 'B1_family_income', 'C1_family_spending'] → sub-poset
  health_prevention: ['A2_health', 'C3_public_health'] → sub-poset
  education_skills: ['A3_education', 'C4_education_childcare'] → sub-poset
  social_wellbeing: ['A4_subjective_wellbeing', 'B2_school_early_childhood', 'B3_safety', 'B4_digital'] → sub-poset

  material_protection: discretising + sub-poset on ['A1_material_conditions', 'B1_family_income', 'C1_family_spending']...
    → 12438480 extensions, score range: [0.188, 0.893]
  health_prevention: discretising + sub-poset on ['A2_health', 'C3_public_health']...
    → 234432 extensions, score range: [0.156, 1.000]
  education_skills: discretising + sub-poset on ['A3_education', 'C4_education_childcare']...
    → 10160640 extensions, score range: [0.250, 1.000]
  social_wellbeing: discretising + sub-poset on ['A4_subjective_wellbeing', 'B2_school_early_ch

In [21]:
n_levels_macro = 3
df_ind_2015 = apply_discretization(
    df_macro_2015,
    columns=list(LEVEL2_GROUPS.keys()),
    n_levels=n_levels_macro,
    normalized_suffix="_norm",
)

df_ind_2018 = apply_discretization(
    df_macro_2018,
    columns=list(LEVEL2_GROUPS.keys()),
    n_levels=n_levels_macro,
    normalized_suffix="_norm",
)

In [22]:
df_ind_2015.write_parquet(f'../data/040_indicators_macro_dim_{n_levels_macro}_level_2015.parquet')
df_ind_2018.write_parquet(f'../data/040_indicators_macro_dim_{n_levels_macro}_level_2018.parquet')

*This notebook is licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*